In [ ]:
print("Car Damage Detection")

Car Damage Detection


In [ ]:
# Import necessary libraries
import os
import zipfile
import requests

# URL to your dataset zip file in the GitHub repository
dataset_url = 'https://raw.githubusercontent.com/rafdimilzano/data/main/cdadataset.zip'  # Change this URL to your dataset location

# Download the dataset
dataset_zip_path = 'cdadataset.zip'
response = requests.get(dataset_url)

# Save the zip file locally
with open(dataset_zip_path, 'wb') as file:
    file.write(response.content)

# Unzip the dataset
with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset')  # Extract the dataset into the 'dataset' folder

# Check the unzipped folder structure
os.listdir('dataset')


['training', 'validation']

In [ ]:
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random as python_random
import tensorflow as tf
import seaborn as sns
import math
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
from tensorflow.keras.callbacks import ModelCheckpoint
from  tensorflow.keras.callbacks import EarlyStopping
from keras import backend as K
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:
# np.random.seed(42)
# tf.random.set_seed(42)

In [ ]:
# Set dataset paths
train_dir = 'dataset/training'    # Path to the Training set
val_dir = 'dataset/validation'    # Path to the Validation set
image_size = (224, 224)  # Resize images to 190x190
batch_size = 32

# Image augmentation for the training set to improve generalization
train_datagen = ImageDataGenerator(rescale=1.0/255)

# No augmentation for validation data, only rescaling
val_datagen = ImageDataGenerator(rescale=1.0/255)

# Loading training data from the structured directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
)

# Loading validation data from the structured directories
validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical'
)


Found 1383 images belonging to 3 classes.
Found 248 images belonging to 3 classes.


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

# Load ResNet50 without the top layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add new layers for transfer learning
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = Dense(1024, activation='relu')(x)
x = Dense(train_generator.num_classes, activation='softmax')(x)

# Create the new model
model = Model(inputs=base_model.input, outputs=x)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print('CNN model set up successfully.')

CNN model set up successfully.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Set training options
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, mode='max')
callbacks_list = [checkpoint]

model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])

print('Training options set.')

Training options set.


In [ ]:
# Train the model
history = model.fit(
    train_generator,
    steps_per_epoch=20,
    epochs=10,
    validation_data=validation_generator,
    callbacks=callbacks_list,
    verbose=1
)

print('Model training completed.')

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 530s 26s/step - accuracy: 0.3668 - loss: 1.2207 - val_accuracy: 0.3669 - val_loss: 1.2093
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 522s 26s/step - accuracy: 0.3799 - loss: 1.1541 - val_accuracy: 0.3669 - val_loss: 1.1447
Epoch 3/10
 4/20 ━━━━━━━━━━━━━━━━━━━━ 6:33 25s/step - accuracy: 0.3919 - loss: 1.1466

/usr/lib/python3.10/contextlib.py:153: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


20/20 ━━━━━━━━━━━━━━━━━━━━ 138s 6s/step - accuracy: 0.3784 - loss: 1.1451 - val_accuracy: 0.3669 - val_loss: 1.1379
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 574s 26s/step - accuracy: 0.3899 - loss: 1.1086 - val_accuracy: 0.3669 - val_loss: 1.1148
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 501s 25s/step - accuracy: 0.3657 - loss: 1.1293 - val_accuracy: 0.3629 - val_loss: 1.1206
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 137s 6s/step - accuracy: 0.4391 - loss: 1.0991 - val_accuracy: 0.3629 - val_loss: 1.1193
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 522s 26s/step - accuracy: 0.3970 - loss: 1.0933 - val_accuracy: 0.3266 - val_loss: 1.1074
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 513s 26s/step - accuracy: 0.4189 - loss: 1.0767 - val_accuracy: 0.2863 - val_loss: 1.1376
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 196s 9s/step - accuracy: 0.4826 - loss: 1.0468 - val_accuracy: 0.2782 - val_loss: 1.1461
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 568s 26s/step - accuracy: 0.3949 - loss: 1.0961 - val_accuracy: 0.3065 - val_lo

In [ ]:
# Predictions on validation dataset
Y_pred_cnn = model.predict(validation_generator)
Y_pred_classes = Y_pred_cnn.argmax(axis=-1)
Y_true = validation_generator.classes

# Performance evaluation (CNN)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_cnn = accuracy_score(Y_true, Y_pred_classes)
precision_cnn = precision_score(Y_true, Y_pred_classes, average='macro')
recall_cnn = recall_score(Y_true, Y_pred_classes, average='macro')
f1_cnn = f1_score(Y_true, Y_pred_classes, average='macro')

print(f'CNN Accuracy: {accuracy_cnn * 100:.2f}%')
print(f'CNN Precision: {precision_cnn:.2f}')
print(f'CNN Recall: {recall_cnn:.2f}')
print(f'CNN F1 Score: {f1_cnn:.2f}')

8/8 ━━━━━━━━━━━━━━━━━━━━ 45s 6s/step
CNN Accuracy: 33.47%
CNN Precision: 0.35
CNN Recall: 0.34
CNN F1 Score: 0.24


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# Feature extraction (you can use a pre-trained model to extract features)
def extract_features(generator, model):
    features = model.predict(generator)
    return np.reshape(features, (features.shape[0], -1))

train_features = extract_features(train_generator, base_model)
validation_features = extract_features(validation_generator, base_model)

# Train k-NN
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(train_features, train_generator.classes)

# Predictions on validation dataset
Y_pred_knn = knn.predict(validation_features)

# Performance evaluation (k-NN)
accuracy_knn = accuracy_score(Y_true, Y_pred_knn)
precision_knn = precision_score(Y_true, Y_pred_knn, average='macro')
recall_knn = recall_score(Y_true, Y_pred_knn, average='macro')
f1_knn = f1_score(Y_true, Y_pred_knn, average='macro')

print(f'k-NN Accuracy: {accuracy_knn * 100:.2f}%')
print(f'k-NN Precision: {precision_knn:.2f}')
print(f'k-NN Recall: {recall_knn:.2f}')
print(f'k-NN F1 Score: {f1_knn:.2f}')

44/44 ━━━━━━━━━━━━━━━━━━━━ 244s 5s/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 5s/step
k-NN Accuracy: 36.29%
k-NN Precision: 0.37
k-NN Recall: 0.36
k-NN F1 Score: 0.36


In [ ]:
from sklearn.svm import SVC

# Train SVM model
svm = SVC(kernel='linear')
svm.fit(train_features, train_generator.classes)

# Predictions on validation dataset
Y_pred_svm = svm.predict(validation_features)

# Performance evaluation (SVM)
accuracy_svm = accuracy_score(Y_true, Y_pred_svm)
precision_svm = precision_score(Y_true, Y_pred_svm, average='macro')
recall_svm = recall_score(Y_true, Y_pred_svm, average='macro')
f1_svm = f1_score(Y_true, Y_pred_svm, average='macro')

print(f'SVM Accuracy: {accuracy_svm * 100:.2f}%')
print(f'SVM Precision: {precision_svm:.2f}')
print(f'SVM Recall: {recall_svm:.2f}')
print(f'SVM F1 Score: {f1_svm:.2f}')

SVM Accuracy: 36.69%
SVM Precision: 0.37
SVM Recall: 0.37
SVM F1 Score: 0.36


In [ ]:
def evaluate_performance(Y_true, Y_pred, categories):
    accuracy = accuracy_score(Y_true, Y_pred)
    precision = precision_score(Y_true, Y_pred, average='macro')
    recall = recall_score(Y_true, Y_pred, average='macro')
    f1 = f1_score(Y_true, Y_pred, average='macro')
    return accuracy, precision, recall, f1